# Day 11. Self-Attention 개념

**오늘의 목표**
- Embedding 직후의 Token 벡터가 **문맥을 모른다**는 문제를 이해한다
- Self-Attention의 핵심 아이디어: **"각 Token의 새 벡터 = 모든 Token 벡터의 가중 평균"** 을 설명할 수 있다
- 가중치가 왜 Softmax(Day 5)로, 왜 Dot Product(Day 2)로 만들어지는지 안다
- `softmax(X @ X.T) @ X` 한 줄을 shape까지 따라갈 수 있다
- 이 단순한 버전의 한계를 보고, 내일 배울 Q/K/V가 왜 필요한지 감을 잡는다

> 진행 방법: 위에서부터 셀을 하나씩 실행(`Shift + Enter`)하세요.
> `# TODO` 셀은 직접 채워보는 연습 문제입니다.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(0)

def r(t, d=3):
    # 텐서를 보기 좋게 반올림한 리스트로
    return [round(v, d) for v in t.tolist()]

def show_weights(w, tokens, title):
    # (T, T) 가중치 표를 히트맵으로 그리기
    plt.figure(figsize=(5.5, 4.5))
    plt.imshow(w, cmap="Blues", vmin=0, vmax=1)
    for i in range(len(tokens)):
        for j in range(len(tokens)):
            v = w[i, j].item()
            plt.text(j, i, f"{v:.2f}", ha="center", va="center",
                     color="white" if v > 0.5 else "black", fontsize=10)
    plt.xticks(range(len(tokens)), tokens)
    plt.yticks(range(len(tokens)), tokens)
    plt.xlabel("참고하는 Token (열)")
    plt.ylabel("지금 Token (행)")
    plt.title(title, fontsize=12)
    plt.colorbar()
    plt.show()

## 1. 문제: Embedding은 문맥을 모른다

Day 8에서 문장을 Token ID로 바꿨고, Day 9에서는 ID마다 벡터 하나를 꺼내 쓰는 **Embedding**을 배웁니다.
그런데 Embedding은 **표에서 꺼내오는 것**이라서 주변에 어떤 단어가 있든 **항상 같은 벡터**가 나옵니다.

In [ ]:
# 아주 작은 가짜 Embedding 표 (단어 → 벡터)
emb = {
    "배가":   torch.tensor([0.5, 0.5, 0.5]),
    "고프다": torch.tensor([0.9, 0.1, 0.0]),
    "바다에": torch.tensor([0.0, 0.2, 0.9]),
    "떴다":   torch.tensor([0.1, 0.0, 0.8]),
}

sentence1 = ["배가", "고프다"]     # 배 = 뱃속
sentence2 = ["바다에", "배가", "떴다"]  # 배 = 선박

print("문장 1의 '배가':", emb[sentence1[0]])
print("문장 2의 '배가':", emb[sentence2[1]])
print("같은 벡터인가?", torch.equal(emb[sentence1[0]], emb[sentence2[1]]))

'배'는 문장 1에서는 **뱃속**, 문장 2에서는 **선박**인데 벡터는 똑같습니다.

커리큘럼의 예시도 같은 문제예요.

```text
철수는 학교에 갔다. 그는 공부를 했다.
```

`그는`의 Embedding은 "3인칭 대명사"라는 정보만 있고, **그게 누구인지(철수)는 모릅니다.**

> **해결하고 싶은 것**: 각 Token이 **주변 Token을 보고** 자기 벡터를 문맥에 맞게 고치게 하자.

## 2. 아이디어: 주변 Token 벡터를 섞는다 (가중 평균)

`그는`이 `철수`를 많이 참고해서 자기 벡터에 `철수`의 정보를 섞으면 됩니다.

```text
새 그는 = 0.60 × 철수는
        + 0.05 × 학교에
        + 0.05 × 갔다
        + 0.20 × 그는      ← 자기 자신도 조금
        + 0.05 × 공부를
        + 0.05 × 했다
```

앞의 숫자들이 **가중치(weight)**: "각 Token을 얼마나 참고할지"입니다.

이해하기 쉽게 4가지 특징을 직접 정해서 벡터를 만들어봅시다.
`[사람, 장소, 행동, 지시어]`

In [ ]:
tokens = ["철수는", "학교에", "갔다", "그는", "공부를", "했다"]
features = ["사람", "장소", "행동", "지시어"]

X = torch.tensor([
    [1.0, 0.0, 0.0, 0.0],   # 철수는: 사람
    [0.0, 1.0, 0.0, 0.0],   # 학교에: 장소
    [0.0, 0.3, 1.0, 0.0],   # 갔다  : 행동 (장소 이동 느낌 조금)
    [0.0, 0.0, 0.0, 1.0],   # 그는  : 지시어 (누군지는 모름!)
    [0.0, 0.2, 0.5, 0.0],   # 공부를: 행동 반, 장소 조금
    [0.0, 0.0, 1.0, 0.0],   # 했다  : 행동
])
T, C = X.shape
print("X shape:", (T, C), " ← Token 6개, 특징 4개")
print("그는 (원래):", dict(zip(features, r(X[3], 2))))

### 2-1. 손으로 정한 가중치로 섞어보기

In [ ]:
w_he = torch.tensor([0.60, 0.05, 0.05, 0.20, 0.05, 0.05])   # '그는'이 각 Token을 보는 비율
print("가중치 합:", round(w_he.sum().item(), 4))

# 방법 1: 반복문으로 하나씩 곱해서 더하기
new_he = torch.zeros(C)
for weight, vec in zip(w_he, X):
    new_he += weight * vec

# 방법 2: 행렬 곱 한 번 (Day 2)  (T,) @ (T, C) → (C,)
new_he_mm = w_he @ X

print()
print("그는 (원래):", dict(zip(features, r(X[3], 3))))
print("그는 (섞은 후):", dict(zip(features, r(new_he, 3))))
print("두 방법이 같은가?", torch.allclose(new_he, new_he_mm))

원래 `그는`은 **사람 = 0**이었는데, 섞은 후에는 **사람 = 0.6**이 됐습니다.
`그는`이 이제 "나는 사람(철수)을 가리킨다"는 정보를 갖게 된 거예요. 이게 Self-Attention이 하는 일의 전부입니다.

> 💡 **Self**-Attention인 이유: 다른 문장이 아니라 **같은 문장(자기 자신) 안의** Token끼리 서로 참고하기 때문입니다.

## 3. 가중치 규칙 → Softmax

가중치는 두 조건을 만족해야 "섞는 비율"이 됩니다.
1. **0 이상** (음수 비율로 섞는 건 이상하니까)
2. **합이 1** (전체 크기가 커지거나 작아지지 않게)

Day 5에서 배운 **Softmax**가 정확히 이 조건을 만들어줍니다.
그래서 먼저 "얼마나 관련 있는지" **점수**를 매기고, Softmax로 **비율**로 바꿉니다.

In [ ]:
scores_he = torch.tensor([3.0, 0.5, 0.5, 1.8, 0.5, 0.5])   # '그는'의 관심 점수 (아무 숫자)
w_from_scores = F.softmax(scores_he, dim=-1)

for t, s, w in zip(tokens, scores_he, w_from_scores):
    print(f"{t:4s}  점수 {s.item():4.1f}  →  가중치 {w.item():.3f}")
print("합계:", round(w_from_scores.sum().item(), 4))

2-1에서 손으로 정한 가중치와 거의 비슷하게 나왔죠?
이제 남은 질문은 하나입니다. **점수는 어디서 오는가?**

## 4. 점수는 어디서? → Dot Product (Day 2)

Day 2에서 **Dot Product가 클수록 두 벡터가 비슷한 방향**이라고 배웠습니다.
가장 단순한 방법: **"나와 비슷한 Token을 많이 참고하자"**

```text
점수(i, j) = X[i] · X[j]
```

모든 Token 쌍을 한 번에 계산하면 Day 2에서 본 그 표가 나옵니다.

```text
scores  = X @ X.T              (T, C) @ (C, T) → (T, T)
weights = softmax(scores)      (T, T)   행마다 합이 1
out     = weights @ X          (T, T) @ (T, C) → (T, C)
```

**입력과 출력의 shape가 똑같습니다.** Token마다 벡터가 하나씩 나오는데, 이제 문맥이 섞인 벡터예요.

In [ ]:
def attention_2d(X):
    # 가장 단순한 Self-Attention: softmax(X Xᵀ) X
    scores = X @ X.T                        # (T, T): 모든 Token 쌍의 유사도
    weights = F.softmax(scores, dim=-1)     # (T, T): 행마다 합이 1
    return weights @ X                      # (T, C): 가중 평균

tokens2 = ["나는", "오늘", "학교에", "간다"]
torch.manual_seed(42)
X2 = torch.randn(4, 8) * 0.5                # 가짜 Embedding: Token 4개 × 길이 8 (값이 크면 Softmax가 한쪽으로 쏠려서 작게. Day 13에서 자세히)

scores2 = X2 @ X2.T
weights2 = F.softmax(scores2, dim=-1)
out2 = weights2 @ X2

print("X       :", tuple(X2.shape))
print("scores  :", tuple(scores2.shape))
print("weights :", tuple(weights2.shape), "  행마다 합 →", r(weights2.sum(dim=-1), 4))
print("out     :", tuple(out2.shape), "  ← 입력과 같은 shape")
print("함수 결과와 같은가?", torch.allclose(out2, attention_2d(X2)))

In [ ]:
show_weights(weights2, tokens2, "가중치 표: softmax(X @ X.T)")

**표 읽는 법**: 행 = 지금 Token, 열 = 참고하는 Token
- `weights[i, j]` = i번째 Token이 새 벡터를 만들 때 j번째 Token을 몇 % 섞는가
- 각 **행**의 합이 1 (100%)

> 💡 대각선(자기 자신)이 유독 진하죠? 이게 8절에서 다룰 **한계**입니다. 기억해두세요.

## 5. 행렬 곱 한 번 = 모든 Token의 가중 평균

`weights @ X`가 정말 "행마다 가중 평균"인지 확인해봅시다.

In [ ]:
i = 2   # '학교에'
manual = torch.zeros(8)
for j in range(4):
    manual += weights2[i, j] * X2[j]

print(f"'{tokens2[i]}'의 가중치:", r(weights2[i], 3))
print("반복문 가중 평균 == out2[2] ?", torch.allclose(manual, out2[i]))

Token 4개 × 4개의 가중 평균을 **행렬 곱 두 번**으로 끝냈습니다.
Token이 1000개여도 코드는 똑같아요. GPU가 행렬 곱을 엄청 빠르게 하기 때문에 Transformer가 효율적인 겁니다.

## 6. 미리보기: 같은 일을 하는 세 가지 방법

GPT는 **앞쪽 Token만** 볼 수 있습니다. (미래 단어를 보면 커닝이니까요. 자세한 건 Day 14 Causal Mask)
가장 단순하게 **"지금까지 나온 Token들의 평균"** 을 세 가지 방법으로 계산해봅시다.
결과가 전부 같다는 것만 확인하면 충분해요.

In [ ]:
torch.manual_seed(1)
B, T, C = 2, 4, 3
x = torch.randn(B, T, C)

# 방법 1: 반복문. t번째 Token = 0 ~ t번째 Token의 평균
out_v1 = torch.zeros(B, T, C)
for b in range(B):
    for t in range(T):
        out_v1[b, t] = x[b, : t + 1].mean(dim=0)

# 방법 2: 아래 삼각형 가중치 행렬 × x
tril = torch.tril(torch.ones(T, T))
w_v2 = tril / tril.sum(dim=1, keepdim=True)      # 행마다 합이 1이 되게
out_v2 = w_v2 @ x                                # (T, T) @ (B, T, C) → (B, T, C)

# 방법 3: 점수 0 → 미래는 -inf → softmax
scores_v3 = torch.zeros(T, T).masked_fill(tril == 0, float("-inf"))
w_v3 = F.softmax(scores_v3, dim=-1)
out_v3 = w_v3 @ x

print("방법 2 가중치:")
print(w_v2)
print()
print("방법 1 == 방법 2 ?", torch.allclose(out_v1, out_v2))
print("방법 1 == 방법 3 ?", torch.allclose(out_v1, out_v3))

- 방법 1은 이해하기 쉽지만 느리고
- 방법 2는 **가중치 표 × 벡터들**이라는 Attention의 모양이 보이고
- 방법 3은 **점수 → Softmax → 가중치** 구조라서, 점수만 바꾸면 "균등 평균"이 아닌 **"중요한 Token을 더 많이"** 섞을 수 있습니다.

실제 GPT는 방법 3에서 점수 자리에 4절의 Dot Product(내일부터는 Q·K)를 넣습니다.
`-inf` 트릭은 Day 5 8-1에서 이미 봤죠? Day 14에서 자세히 다룹니다.

## 7. Batch 버전: `(B, T, C)`

문장 여러 개를 한 번에 처리하려면 `.T` 대신 **마지막 두 축만** 뒤집으면 됩니다. (Day 2 5-3)

In [ ]:
B, T, C = 2, 4, 8
xb = torch.randn(B, T, C)

scores_b = xb @ xb.transpose(-2, -1)     # (B, T, C) @ (B, C, T) → (B, T, T)
weights_b = F.softmax(scores_b, dim=-1)  # (B, T, T)
out_b = weights_b @ xb                   # (B, T, T) @ (B, T, C) → (B, T, C)

print("x       :", tuple(xb.shape))
print("scores  :", tuple(scores_b.shape), " ← 문장마다 (T×T) 표가 하나씩")
print("out     :", tuple(out_b.shape))

## 8. 한계: 이 단순한 버전으로는 '철수'를 못 찾는다

2절의 철수 문장에 4절의 방법(`softmax(X Xᵀ)`)을 그대로 적용해봅시다.

In [ ]:
weights_cs = F.softmax(X @ X.T, dim=-1)
show_weights(weights_cs, tokens, "softmax(X @ X.T): 그는 → ?")

print("'그는'의 가중치:")
for t, w in zip(tokens, weights_cs[3]):
    print(f"  {t:4s} {w.item():.3f}")

`그는`은 **자기 자신만** 많이 보고, `철수는`은 다른 단어들과 똑같이 0.13밖에 안 봅니다. 왜 그럴까요?

**문제 1. 자기 자신이 항상 제일 비슷하다**
`X[i] · X[i]`는 자기 벡터의 길이 제곱이라 보통 가장 큽니다. 대각선이 진했던 이유예요.

**문제 2. 벡터 하나가 세 가지 역할을 동시에 한다**

| 역할 | 질문 | `그는` 입장에서 |
|---|---|---|
| 찾는 쪽 | "나는 어떤 정보가 필요하지?" | "내가 가리키는 **사람**이 필요해" |
| 찾아지는 쪽 | "나는 어떤 정보를 갖고 있지?" | "나는 지시어야" |
| 전해주는 내용 | "나를 참고하면 무엇을 줄까?" | "3인칭이라는 정보" |

`그는`의 '찾는 쪽' 벡터는 **사람** 방향이어야 `철수는`과 Dot Product가 커집니다.
그런데 지금은 벡터가 하나뿐이라 `그는` 벡터는 **지시어** 방향이고, 그래서 `철수는`을 못 찾아요.

> **해결책 (내일 Day 12)**: 같은 X에서 **역할별로 다른 벡터 3개**를 만듭니다.
> - **Query**: 찾는 쪽 (내가 원하는 것)
> - **Key**: 찾아지는 쪽 (내가 가진 특징)
> - **Value**: 전해주는 내용
>
> 만드는 방법은 Day 3에서 배운 **Linear Layer** 세 개예요.

## 9. LLM과의 연결

GPT 안에서 Token끼리 정보를 주고받는 곳은 **Self-Attention 단 한 곳**입니다.

```text
Token ID
  ↓  Embedding          각 Token이 자기 벡터만 가짐 (문맥 모름)
  ↓  Self-Attention     ← 오늘: 다른 Token 정보를 섞음 (Token끼리 대화)
  ↓  FFN (Day 4)        각 Token을 따로따로 처리 (Token끼리 대화 없음)
  ↓  ... × N층
  ↓  lm_head → Softmax → 다음 Token
```

FFN은 `(B, T, C)`의 마지막 축만 바꾸기 때문에 Token끼리 섞이지 않아요. (Day 3 5절, Day 4 4절)
**T 축을 가로질러 정보를 섞는 건 Attention뿐**입니다. 그래서 Transformer 논문 제목이 *"Attention Is All You Need"* 예요.

## 10. 연습 문제

`assert`가 에러 없이 통과하면 정답입니다.

**Q1.** 아래 세 벡터를 가중치 `[0.5, 0.25, 0.25]`로 섞은 결과를 **손으로** 계산해서 적으세요.

```text
v1 = [1, 0]
v2 = [0, 1]
v3 = [1, 1]
```

In [ ]:
V = torch.tensor([[1.0, 0.0],
                  [0.0, 1.0],
                  [1.0, 1.0]])
w = torch.tensor([0.5, 0.25, 0.25])

my_answer = "?"   # TODO: 리스트로. 예) [0.1, 0.2]

assert my_answer != "?", "TODO를 채워주세요"
assert torch.allclose(torch.tensor(my_answer, dtype=torch.float32), w @ V), "가중치 × 벡터를 하나씩 더해보세요"
print("Q1 정답!")

**Q2.** 아래 세 가중치 표 중 **Self-Attention 가중치로 쓸 수 있는 것**은? `"A"`, `"B"`, `"C"` 중 하나를 적으세요.

In [ ]:
A = torch.tensor([[0.7, 0.3, 0.0],
                  [0.2, 0.5, 0.3],
                  [0.1, 0.1, 0.8]])
B_ = torch.tensor([[0.9, 0.3, -0.2],
                   [0.2, 0.5, 0.3],
                   [0.1, 0.1, 0.8]])
C_ = torch.tensor([[0.7, 0.3, 0.2],
                   [0.2, 0.5, 0.3],
                   [0.1, 0.1, 0.8]])

print("A 행 합:", r(A.sum(dim=-1), 2))
print("B 행 합:", r(B_.sum(dim=-1), 2))
print("C 행 합:", r(C_.sum(dim=-1), 2))

my_choice = "?"   # TODO: "A", "B", "C" 중 하나

def valid(w):
    return bool((w >= 0).all() and torch.allclose(w.sum(dim=-1), torch.ones(w.shape[0])))

assert my_choice in ("A", "B", "C"), "A, B, C 중 하나를 적어주세요"
assert valid({"A": A, "B": B_, "C": C_}[my_choice]), "가중치가 만족해야 하는 두 조건을 다시 확인해보세요"
print("Q2 정답!")

**Q3.** 4절의 `attention_2d`는 `(T, C)` 하나만 받습니다.
**`(B, T, C)`** 를 받는 `simple_self_attention`을 완성하세요.

힌트: 7절을 참고하세요. `.T` 대신 `transpose(-2, -1)`

In [ ]:
def simple_self_attention(X):
    return None  # TODO: softmax(X Xᵀ) X  를 batch 버전으로

X_test = torch.randn(3, 5, 8)
out = simple_self_attention(X_test)

assert out is not None, "TODO를 채워주세요"
assert tuple(out.shape) == (3, 5, 8), "출력 shape가 입력과 같아야 해요"
expected = torch.stack([attention_2d(X_test[b]) for b in range(3)])
assert torch.allclose(out, expected, atol=1e-6), "문장마다 attention_2d를 한 결과와 달라요"
print("Q3 정답!")

**Q4.** `X`의 shape가 `(B=3, T=7, C=16)`일 때, 아래 세 텐서의 shape를 실행하기 **전에** 예측하세요.

```python
scores  = X @ X.transpose(-2, -1)
weights = F.softmax(scores, dim=-1)
out     = weights @ X
```

In [ ]:
answers = {
    "scores":  "?",   # TODO: 튜플로
    "weights": "?",   # TODO
    "out":     "?",   # TODO
}

X = torch.randn(3, 7, 16)
scores = X @ X.transpose(-2, -1)
weights = F.softmax(scores, dim=-1)
out = weights @ X
checks = {"scores": tuple(scores.shape), "weights": tuple(weights.shape), "out": tuple(out.shape)}

for k in checks:
    mark = "✅" if answers[k] == checks[k] else "❌"
    print(f"{mark} {k:8s} 내 답: {answers[k]}")
assert answers == checks, "❌ 표시된 것을 다시 생각해보세요"
print("Q4 정답!")

## 11. 체크 질문 (코드 없이 말로 답해보기)

답을 아래 셀에 적어두거나 채팅으로 알려주세요.

1. Embedding 직후의 Token 벡터에는 어떤 한계가 있는가? (`그는`, `배` 예시로)
2. Self-Attention의 출력은 입력 벡터들의 무엇인가?
3. 가중치가 만족해야 하는 두 조건은? 이를 위해 어떤 함수를 쓰는가?
4. `X @ X.T`의 shape는? 각 칸은 무엇을 의미하는가?
5. `softmax(X Xᵀ) X`만으로는 `그는`이 `철수`를 잘 찾지 못한다. 왜 그런가?
6. GPT에서 Token끼리 정보를 주고받는 곳은 어디인가? FFN과 무엇이 다른가?

**내 답:**

1.
2.
3.
4.
5.
6.